# Notebook 9: t-SNE — t-Distributed Stochastic Neighbour Embedding

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain t-SNE's core idea: modelling neighbourhood similarities as probabilities in both high-D and low-D space
2. Describe the role of **perplexity** and what happens when you change it
3. Articulate the **crowding problem** and explain how the Student-t distribution addresses it
4. Apply t-SNE correctly — for **visualization only**, not for distance preservation or downstream modelling

---

**Week 11 Theme:** Customer Segmentation / Retail Data

**Prerequisites:** PCA (Notebook 07), KMeans (Notebook 01-03)

## The Analogy — Before Any Math

Imagine you're trying to fit a map of a **100-country world onto a single page**.

- **PCA** is like a **satellite photo** — it preserves large-scale geography faithfully. You can see the big continents in the right positions relative to each other. But small, closely-packed countries get squished together and you can't see their internal detail.

- **t-SNE** is like a **custom hand-drawn map** — it sacrifices long-range accuracy to make sure *nearby countries (similar data points) look close together*. Neighbouring villages will always appear near each other. But don't try to measure the distance from Europe to Asia on this map — it's been distorted to serve local accuracy.

The key insight: **t-SNE is obsessively focused on local neighbourhood structure**. It does one job and does it very well: making clusters pop visually.

## Why Does This Matter for ML?

High-dimensional data is everywhere:

| Domain | What's high-dimensional? | Typical dimensions |
|--------|--------------------------|--------------------|
| NLP | Word embeddings, sentence vectors | 300–1536 |
| Computer Vision | CNN activations, image pixels | 512–4096 |
| Genomics | Gene expression profiles | 20,000+ |
| Retail | Customer behaviour vectors | 50–500 |

We can't visualise more than 3 dimensions. t-SNE is the **gold standard tool** for compressing high-dimensional structure into a 2D plot that humans can interpret.

**Concrete uses:**
- Checking whether your learned embeddings have meaningful clusters (NLP, recommender systems)
- Exploring whether customer segments are well-separated before running KMeans
- Debugging neural network representations — do activations cluster by class?
- Validating that dimensionality reduction hasn't destroyed important structure

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import time
import warnings

warnings.filterwarnings('ignore')       # suppress convergence warnings for cleaner output
np.random.seed(42)                       # reproducibility for all numpy operations
sns.set_theme(style='whitegrid')         # consistent seaborn visual style

print("All imports successful.")
print(f"NumPy version: {np.__version__}")

## The Core Idea: Similarities as Probabilities

t-SNE does something elegant: it turns **distances into probabilities**, so it can reason about "how likely are these two points to be neighbours?"

### Step 1 — High-dimensional space: Gaussian similarities

For every pair of points $i, j$ in the original (high-dimensional) space, t-SNE computes a **conditional probability**:

> "Given that I'm standing at point $i$, what's the probability that point $j$ is my neighbour?"

$$p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}$$

Think of it as: nearby points get high probability, distant points get near-zero probability. $\sigma_i$ controls the neighbourhood radius (it's set automatically based on **perplexity** — more on this soon).

We symmetrize: $p_{ij} = (p_{j|i} + p_{i|j}) / 2n$

### Step 2 — Low-dimensional space: Student-t similarities

In 2D, t-SNE uses a **Student-t distribution** (with 1 degree of freedom, i.e. a Cauchy distribution) instead of a Gaussian:

$$q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l}(1 + \|y_k - y_l\|^2)^{-1}}$$

Why Student-t? Its **heavier tails** allow moderately-similar points to be spread further apart in 2D without being penalized — this solves the crowding problem (see below).

### Step 3 — Optimize: minimize KL divergence

t-SNE moves the 2D points $y_i$ around to make $Q$ match $P$ as closely as possible:

$$\text{Minimize} \quad KL(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

In plain English: **if two points are close in the original space ($p_{ij}$ is high), they should also be close in 2D ($q_{ij}$ should also be high)**. Violating this is very costly. But the reverse (far-apart points in high-D being placed near each other in 2D) is penalized less — which is why t-SNE can distort global distances.

In [ ]:
np.random.seed(42)

# ── Load the digits dataset (8x8 pixel images of handwritten 0-9 digits)
# This is a standard proxy for MNIST — 1797 samples, 64 features each
digits = load_digits()
X_digits = digits.data          # shape: (1797, 64) — each sample is a flattened 8x8 image
y_digits = digits.target        # shape: (1797,)   — digit labels 0-9

print(f"Dataset shape: {X_digits.shape}")
print(f"Number of classes: {len(np.unique(y_digits))}")

# ── Apply PCA to 2D
pca = PCA(n_components=2, random_state=42)
X_pca_2d = pca.fit_transform(X_digits)
print(f"PCA explained variance (2 components): {pca.explained_variance_ratio_.sum():.1%}")

# ── Apply t-SNE to 2D (perplexity=30 is the standard default)
print("Running t-SNE... (may take ~30 seconds)")
t0 = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne_2d = tsne.fit_transform(X_digits)
print(f"t-SNE completed in {time.time()-t0:.1f}s")

# ── Plot PCA vs. t-SNE side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Use a discrete colormap for 10 classes
cmap = plt.cm.get_cmap('tab10', 10)

for ax, X_2d, title in zip(axes,
                            [X_pca_2d, X_tsne_2d],
                            ['PCA (2 components, 28.5% variance)',
                             't-SNE (perplexity=30)']):
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                         c=y_digits, cmap=cmap,
                         s=10, alpha=0.7)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')

# Shared colorbar showing digit class labels
cbar = fig.colorbar(scatter, ax=axes, orientation='vertical',
                    fraction=0.03, pad=0.04)
cbar.set_label('Digit class', fontsize=11)
cbar.set_ticks(np.arange(10))

fig.suptitle('PCA vs. t-SNE on Digits Dataset', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("Observation: PCA shows some separation but classes heavily overlap.")
print("t-SNE produces tight, well-separated clusters for each digit — even visually similar ones (3 vs 8, 4 vs 9).")

## The Perplexity Parameter

**Perplexity** is the single most important hyperparameter in t-SNE. Here's what it controls:

> Perplexity ≈ the **effective number of neighbours** each point considers when computing similarities.

More precisely, perplexity determines the bandwidth $\sigma_i$ of the Gaussian kernel. A high perplexity means a wider Gaussian — each point considers more neighbours, capturing more global structure.

| Perplexity | Neighbourhood size | Effect on plot |
|------------|-------------------|----------------|
| 5 | Very small (local only) | Tight isolated islands, may fragment real clusters |
| 15 | Small-medium | Local structure with some cohesion |
| 30 | Medium (default) | Good balance — use this first |
| 50 | Medium-large | Smoother, more connected clusters |
| 100 | Large (global) | Very smooth, may merge distinct clusters |

**Rule of thumb:** Use 5–50. For datasets with fewer than 100 points, perplexity must be less than n/3. The default of 30 works well for most datasets.

**Important gotcha:** Always run t-SNE to convergence (increase `n_iter` if needed) before interpreting results.

In [ ]:
np.random.seed(42)

# Use a subset of 500 points for speed — the perplexity effect is clear even at this scale
subset_idx = np.random.choice(len(X_digits), size=500, replace=False)
X_sub = X_digits[subset_idx]
y_sub = y_digits[subset_idx]

perplexities = [5, 15, 30, 50, 100]   # five different perplexity values to compare
tsne_results = {}

for p in perplexities:
    print(f"Running t-SNE with perplexity={p}...")
    # n_iter=500 is enough for a comparison demo; production runs use 1000+
    tsne_model = TSNE(n_components=2, perplexity=p, random_state=42, n_iter=500)
    tsne_results[p] = tsne_model.fit_transform(X_sub)

# ── Plot all 5 results
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
cmap = plt.cm.get_cmap('tab10', 10)

for ax, p in zip(axes, perplexities):
    X_2d = tsne_results[p]
    ax.scatter(X_2d[:, 0], X_2d[:, 1],
               c=y_sub, cmap=cmap,
               s=15, alpha=0.8)
    ax.set_title(f'Perplexity = {p}', fontsize=12)
    ax.set_xticks([])    # remove axis numbers — t-SNE coordinates are arbitrary
    ax.set_yticks([])

fig.suptitle('Effect of Perplexity on t-SNE Layout (Digits, n=500)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("What you should see:")
print("  Perplexity=5:   Fragmented — clusters split into sub-islands, even within one digit class")
print("  Perplexity=15:  Better coherence but still some fragmentation")
print("  Perplexity=30:  Clean clusters — good default choice")
print("  Perplexity=50:  Slightly smeared but still recognisable clusters")
print("  Perplexity=100: Very smooth, clusters start merging — loses fine structure")

## The Crowding Problem — And How t-SNE Solves It

Here's a fundamental geometric fact that causes problems when reducing dimensionality:

> In high dimensions, **many points can be equidistant from each other**. In 2D, you simply can't fit many equidistant points without crowding.

Think about it physically: on a 1D line, you can have at most 2 points equidistant from a center point. On a 2D plane, 6 points. In 3D, you can arrange 12 roughly equidistant points around a sphere. In 10D... hundreds.

**The problem:** If a high-dimensional point has 50 genuine neighbours, mapping to 2D means only ~6 of them can stay close. The rest get crowded to the periphery — creating **artificial distances that don't reflect the original structure**.

**t-SNE's fix:** Use a Student-t distribution (heavier tails) in the low-dimensional space. This distribution allows **moderately-similar points to be placed further apart** without being heavily penalized — giving them room to spread out and avoid crowding.

In practical terms: the heavier tail of the Student-t vs. Gaussian means that a small $q_{ij}$ (points placed far apart in 2D) is "less surprising" and incurs lower KL cost. This effectively "inflates" the 2D space and reduces crowding.

In [ ]:
np.random.seed(42)

# ── Demonstrate that in high-D, all pairwise distances look similar
# Generate points uniformly on the surface of a 10-dimensional sphere
n_points = 200
n_dims_high = 10

# Standard way to sample from a sphere: normalise Gaussian vectors
X_high = np.random.randn(n_points, n_dims_high)      # Gaussian random
X_high = X_high / np.linalg.norm(X_high, axis=1, keepdims=True)  # project onto unit sphere

# Do the same for a 2D sphere (circle)
X_low = np.random.randn(n_points, 2)
X_low = X_low / np.linalg.norm(X_low, axis=1, keepdims=True)  # project onto unit circle

# Compute all pairwise distances
dists_high = pairwise_distances(X_high).flatten()    # upper-triangle + diagonal
dists_low  = pairwise_distances(X_low).flatten()

# Remove self-distances (zeros on the diagonal)
dists_high = dists_high[dists_high > 0]
dists_low  = dists_low[dists_low > 0]

# ── Compute coefficient of variation (std/mean) — a measure of relative spread
cv_high = dists_high.std() / dists_high.mean()
cv_low  = dists_low.std()  / dists_low.mean()

# ── Plot histograms of pairwise distances
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(dists_high, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Pairwise distances on 10D sphere\n(CV = {cv_high:.3f} — distances concentrate!)',
                  fontsize=11)
axes[0].set_xlabel('Euclidean distance')
axes[0].set_ylabel('Count')
axes[0].axvline(dists_high.mean(), color='red', linestyle='--', label=f'Mean = {dists_high.mean():.2f}')
axes[0].legend()

axes[1].hist(dists_low, bins=40, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Pairwise distances on 2D circle\n(CV = {cv_low:.3f} — much more spread)',
                  fontsize=11)
axes[1].set_xlabel('Euclidean distance')
axes[1].set_ylabel('Count')
axes[1].axvline(dists_low.mean(), color='red', linestyle='--', label=f'Mean = {dists_low.mean():.2f}')
axes[1].legend()

fig.suptitle('The Crowding Problem: High-D Distances Concentrate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print(f"10D sphere: mean distance = {dists_high.mean():.3f}, std = {dists_high.std():.3f}")
print(f"2D circle:  mean distance = {dists_low.mean():.3f}, std = {dists_low.std():.3f}")
print()
print("Key insight: In 10D, almost ALL pairs of points have nearly the SAME distance (~1.4).")
print("This means 'neighbour' vs 'non-neighbour' is almost indistinguishable by distance alone.")
print("In 2D, distances vary much more widely — which is exactly the space we're projecting into.")
print("This mismatch is the crowding problem. t-SNE's heavier-tailed q_ij absorbs this mismatch.")

## t-SNE vs. PCA: A Comparison Table

| Property | PCA | t-SNE |
|----------|-----|-------|
| **What it preserves** | Global variance (large-scale structure) | Local neighbourhood structure |
| **Global structure** | ✅ Preserved (large clusters in right positions) | ❌ Distorted (cluster positions are arbitrary) |
| **Local structure** | ⚠️ Mediocre (linear projections miss local clusters) | ✅ Excellent |
| **Cluster distances meaningful?** | ✅ Yes (larger PC distances = more different) | ❌ **No** — never interpret inter-cluster distances |
| **Cluster sizes meaningful?** | ✅ Yes | ❌ **No** — cluster size in t-SNE is arbitrary |
| **Deterministic?** | ✅ Yes (unique solution) | ❌ No (random initialisation → different layouts each run) |
| **Can embed new points?** | ✅ Yes (`transform()`) | ❌ No (must rerun on full dataset) |
| **Computational cost** | O(n·d²) — fast | O(n² log n) — slow for large n |
| **Interpretability** | ✅ Loadings have meaning | ❌ Axes are meaningless |
| **Best use case** | Feature reduction, preprocessing, large n | Visualization of local cluster structure |

### The Golden Rule of t-SNE:

> **Only use t-SNE for visualization. Never use t-SNE as a feature extraction step for downstream models.**

The reason: t-SNE actively distorts global structure and produces non-deterministic results. A classifier trained on t-SNE features would fail on new data.

In [ ]:
np.random.seed(42)

# ── Demonstrate that t-SNE layouts differ across random seeds
# Use a small subset (300 points) and 3 different seeds
subset_idx = np.random.choice(len(X_digits), size=300, replace=False)
X_sub = X_digits[subset_idx]
y_sub = y_digits[subset_idx]

seeds = [0, 7, 99]   # three different random seeds
results_by_seed = {}

for seed in seeds:
    print(f"Running t-SNE with seed={seed}...")
    tsne_s = TSNE(n_components=2, perplexity=30, random_state=seed, n_iter=500)
    results_by_seed[seed] = tsne_s.fit_transform(X_sub)

# ── Plot the three layouts
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
cmap = plt.cm.get_cmap('tab10', 10)

for ax, seed in zip(axes, seeds):
    X_2d = results_by_seed[seed]
    ax.scatter(X_2d[:, 0], X_2d[:, 1],
               c=y_sub, cmap=cmap, s=20, alpha=0.85)
    ax.set_title(f'Seed = {seed}', fontsize=12)
    ax.set_xticks([])   # t-SNE axes are not interpretable numbers
    ax.set_yticks([])

fig.suptitle('t-SNE is Non-Deterministic: Same Data, Three Random Seeds\n'
             '(Cluster POSITIONS change, but within-cluster structure is consistent)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("What you should observe:")
print("  - The global arrangement (which cluster is top-left vs. bottom-right) changes each run")
print("  - But: points of the same class (digit) still cluster tightly together across runs")
print("  - Conclusion: within-cluster cohesion is meaningful; between-cluster positions are NOT")
print()
print("WARNING: Never say 'cluster A is far from cluster B in t-SNE' — that distance is arbitrary!")

In [ ]:
np.random.seed(42)

# ── Generate synthetic retail customer data (Week 11 theme)
# 5 customer segments, 5 features: recency, frequency, monetary, age, online_ratio
n_customers_per_segment = 120

# Segment definitions (mean feature values)
segment_params = {
    'Budget Shoppers':  {'recency': 15, 'frequency': 8,  'monetary': 120,  'age': 45, 'online_ratio': 0.2},
    'Loyal Regulars':   {'recency': 5,  'frequency': 20, 'monetary': 350,  'age': 38, 'online_ratio': 0.5},
    'Premium Buyers':   {'recency': 8,  'frequency': 12, 'monetary': 900,  'age': 52, 'online_ratio': 0.4},
    'Digital Natives':  {'recency': 3,  'frequency': 25, 'monetary': 280,  'age': 28, 'online_ratio': 0.95},
    'Churned Customers':{'recency': 90, 'frequency': 2,  'monetary': 80,   'age': 55, 'online_ratio': 0.15},
}

all_customers = []
all_labels    = []

for seg_name, params in segment_params.items():
    # Add Gaussian noise around each segment centre
    segment_data = np.column_stack([
        np.random.normal(params['recency'],      params['recency'] * 0.3,      n_customers_per_segment),
        np.random.normal(params['frequency'],    params['frequency'] * 0.25,   n_customers_per_segment),
        np.random.normal(params['monetary'],     params['monetary'] * 0.3,     n_customers_per_segment),
        np.random.normal(params['age'],          5,                            n_customers_per_segment),
        np.clip(np.random.normal(params['online_ratio'], 0.1, n_customers_per_segment), 0, 1),
    ])
    all_customers.append(segment_data)
    all_labels.extend([seg_name] * n_customers_per_segment)

X_retail = np.vstack(all_customers)           # shape: (600, 5)
y_retail  = np.array(all_labels)              # shape: (600,)  — segment labels

# Standardise features (important before t-SNE — it's sensitive to scale)
scaler = StandardScaler()
X_retail_scaled = scaler.fit_transform(X_retail)

# ── PCA 2D projection
pca_retail = PCA(n_components=2, random_state=42)
X_retail_pca = pca_retail.fit_transform(X_retail_scaled)

# ── t-SNE 2D projection
print("Running t-SNE on retail data...")
tsne_retail = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_retail_tsne = tsne_retail.fit_transform(X_retail_scaled)

# ── Side-by-side comparison
seg_labels = list(segment_params.keys())
color_map  = dict(zip(seg_labels, plt.cm.Set1.colors[:5]))
colors     = [color_map[label] for label in y_retail]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, X_2d, title in zip(axes,
                            [X_retail_pca, X_retail_tsne],
                            ['PCA (2D) — Retail Customers',
                             't-SNE (perplexity=30) — Retail Customers']):
    for seg in seg_labels:
        mask = y_retail == seg
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[color_map[seg]], label=seg, s=20, alpha=0.75)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

axes[1].legend(loc='lower left', fontsize=9, markerscale=2)
fig.suptitle('Customer Segments: PCA vs. t-SNE Visualisation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("t-SNE should reveal tighter, better-separated customer clusters than PCA.")
print("This is especially true for segments that are close in PCA space but locally distinct.")

## Computational Cost

t-SNE's main weakness is **speed**:

| Algorithm variant | Complexity | When to use |
|-------------------|-----------|-------------|
| Naive t-SNE | O(n²) | n < 5,000 |
| Barnes-Hut t-SNE (sklearn default) | O(n² log n) | n < 100,000 |
| FIt-SNE | O(n log n) | n > 100,000 (external library) |

Rough timing benchmarks (modern laptop, sklearn):

| n | Barnes-Hut t-SNE |
|---|------------------|
| 1,000 | ~5 seconds |
| 5,000 | ~30 seconds |
| 10,000 | ~2–3 minutes |
| 100,000 | ~30–60 minutes |

**Practical tip:** For large datasets, first reduce to 50 dimensions with PCA, then apply t-SNE. This:
1. Removes noise dimensions (improving quality)
2. Dramatically speeds up t-SNE (fewer dimensions = faster pairwise distances)
3. Preserves global structure that PCA is good at, then t-SNE refines local structure

In [ ]:
np.random.seed(42)

# ── Benchmark t-SNE on the digits dataset (1797 samples, 64 features)
# We'll time t-SNE on the full dataset
print(f"Dataset: {X_digits.shape[0]} samples, {X_digits.shape[1]} features")
print()

print("Timing t-SNE on full digits dataset (n=1797, d=64)...")
t0 = time.time()
tsne_full = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne_full = tsne_full.fit_transform(X_digits)
t_full = time.time() - t0
print(f"t-SNE on 64 features: {t_full:.2f} seconds")

# Colour points by digit class for the final plot
cmap = plt.cm.get_cmap('tab10', 10)
fig, ax = plt.subplots(figsize=(9, 7))

scatter = ax.scatter(X_tsne_full[:, 0], X_tsne_full[:, 1],
                     c=y_digits, cmap=cmap,
                     s=12, alpha=0.8)

# Annotate each cluster with its digit label
for digit in range(10):
    mask = y_digits == digit
    cx = X_tsne_full[mask, 0].mean()     # centroid x
    cy = X_tsne_full[mask, 1].mean()     # centroid y
    ax.text(cx, cy, str(digit), fontsize=14, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

ax.set_title(f't-SNE on Full Digits Dataset (n=1797, time={t_full:.1f}s)',
             fontsize=13, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
plt.colorbar(scatter, ax=ax, label='Digit class')
plt.tight_layout()
plt.show()

## How to Use t-SNE Correctly: A Practical Checklist

t-SNE is powerful but easy to misuse. Follow these guidelines:

### ✅ Do:

1. **Reduce dimensions with PCA first** — Run PCA to 50 dimensions before t-SNE. This removes noise and speeds things up without losing meaningful structure.

2. **Try multiple random seeds** — Run t-SNE 3-5 times. If the same cluster structure appears consistently, it's real. If it's wildly different every run, be cautious.

3. **Vary perplexity** — Try 5, 30, and 50. If a cluster appears at all perplexities, it's likely genuine.

4. **Standardise features** — t-SNE uses Euclidean distances. Features on different scales will dominate the similarity computation.

5. **Run to convergence** — Use at least `n_iter=1000`. Check that the KL divergence has stabilised.

### ❌ Don't:

1. **Don't interpret cluster distances** — Cluster A being far from cluster B in t-SNE means nothing.

2. **Don't interpret cluster sizes** — A large cluster in t-SNE is not necessarily a large cluster in the original space.

3. **Don't use t-SNE coordinates as input features** — They will give different results on new data (no `transform()` method).

4. **Don't run KMeans on t-SNE output** — Cluster in the original (or PCA-reduced) space; use t-SNE only for visualization.

5. **Don't run t-SNE on > 100k points** — Use UMAP or FIt-SNE for large datasets.

In [ ]:
np.random.seed(42)

# ── Best practice: PCA(50) → t-SNE(2) vs. raw t-SNE(2)
# On the digits dataset, d=64 so PCA(50) still keeps almost all variance

# Step 1: PCA to 50 dimensions
pca_50 = PCA(n_components=50, random_state=42)
X_pca50 = pca_50.fit_transform(X_digits)
print(f"PCA(50) explains {pca_50.explained_variance_ratio_.sum():.1%} of variance")

# Step 2a: t-SNE on raw 64-D data
print("Running t-SNE on 64-D data...")
t0 = time.time()
tsne_raw = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne_raw = tsne_raw.fit_transform(X_digits)
time_raw = time.time() - t0

# Step 2b: t-SNE on PCA-reduced 50-D data
print("Running t-SNE on PCA(50) data...")
t0 = time.time()
tsne_pca = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne_pca = tsne_pca.fit_transform(X_pca50)
time_pca = time.time() - t0

print(f"t-SNE on 64-D:  {time_raw:.2f}s  |  KL divergence = {tsne_raw.kl_divergence_:.4f}")
print(f"t-SNE on 50-D:  {time_pca:.2f}s  |  KL divergence = {tsne_pca.kl_divergence_:.4f}")

# ── Side-by-side plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cmap = plt.cm.get_cmap('tab10', 10)

for ax, X_2d, title, t in zip(axes,
                               [X_tsne_raw, X_tsne_pca],
                               ['Raw t-SNE (64-D → 2-D)',
                                'Best-Practice: PCA(50) → t-SNE(2)'],
                               [time_raw, time_pca]):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                    c=y_digits, cmap=cmap, s=10, alpha=0.8)
    ax.set_title(f'{title}\n(time: {t:.1f}s)', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(sc, ax=axes, label='Digit class', fraction=0.03, pad=0.04)
fig.suptitle('Best Practice: PCA Pre-Processing Improves t-SNE Quality',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("Lower KL divergence = better fit. PCA pre-processing typically yields a lower KL divergence")
print("because noise dimensions are removed before t-SNE computes similarities.")

## Self-Check Questions

Test your understanding. Try to answer each question before revealing the answer.

---

**Question 1:** You run t-SNE on a dataset of customer embeddings and see that "Segment A" and "Segment B" are very far apart in the 2D plot, while "Segment A" and "Segment C" are close. Can you conclude that Segments A and B are more different from each other than Segments A and C?

<details><summary>Show answer</summary>

**No.** Inter-cluster distances in t-SNE are not meaningful. t-SNE optimises to preserve *local neighbourhood structure* (are nearby points still nearby?) but actively distorts global distances. The fact that A and B are far apart in the plot tells you nothing reliable about how different they are in the original high-dimensional space. To compare cluster similarity, compute distances in the original feature space (or in PCA-reduced space).

</details>

---

**Question 2:** You run t-SNE with perplexity=5 and see 20 isolated micro-clusters. You run it again with perplexity=50 and see 4 large merged blobs. Which result is "correct"?

<details><summary>Show answer</summary>

Neither result alone is "correct" — they're showing different scales of structure. Perplexity=5 is over-emphasising local structure, fragmenting real clusters. Perplexity=50 may be over-smoothing, merging genuinely distinct clusters. The right approach is to **try multiple perplexity values** (5, 15, 30, 50) and look for structures that are **consistent across all settings** — those are the most reliable. If a cluster appears at all perplexity values, it's likely real.

</details>

---

**Question 3:** A colleague wants to use t-SNE to reduce a customer dataset from 200 dimensions to 2 dimensions, then train a KMeans clustering model on the 2D output. What's wrong with this approach, and what should they do instead?

<details><summary>Show answer</summary>

Two problems:

1. **t-SNE distorts global structure** — cluster sizes and inter-cluster distances in the 2D output are not representative of the original space. KMeans on t-SNE output would produce clusters that look good visually but may not correspond to meaningful groups in the original data.

2. **t-SNE has no `transform()` method** — there is no way to embed new customer data into the same 2D space. The KMeans model trained on these 2D coordinates would be unusable for new customers.

**What to do instead:** Run KMeans (or another clustering algorithm) directly in the original 200-D space, or first reduce with PCA to ~20–50 dimensions. Then use t-SNE *only to visualize* the resulting clusters — colour the t-SNE plot by the cluster labels from KMeans.

</details>

## Key Takeaways

1. **t-SNE converts distances to probabilities** — $p_{ij}$ in high-D (Gaussian kernel) and $q_{ij}$ in low-D (Student-t kernel). It minimises $KL(P \| Q)$ via gradient descent.

2. **Perplexity ≈ effective neighbourhood size** — Start with 30. Lower values reveal finer local structure; higher values give a smoother, more global picture.

3. **The crowding problem** — High-D data has too many equidistant neighbours to fit in 2D. The Student-t distribution's heavy tails solve this by allowing moderate-similarity points to spread out further in the low-D map.

4. **What t-SNE preserves and what it doesn't:**
   - ✅ Local neighbourhood structure (within-cluster tightness)
   - ❌ Inter-cluster distances (meaningless in t-SNE)
   - ❌ Cluster sizes (not proportional to real sizes)

5. **Best practice pipeline:** Standardise → PCA(50) → t-SNE(2). Use multiple seeds. Visualise only.

6. **t-SNE is slow** — O(n² log n). Not suitable for n > 100k points. Use UMAP for large datasets.

---

## Up Next: Notebook 10 — UMAP

UMAP is t-SNE's faster, more globally-aware successor. It solves t-SNE's key limitations:
- Much faster (O(n^1.14) empirically)
- Preserves global structure better
- Has a `transform()` method for new data
- Can be used as a feature extractor (with care)

We'll explore whether UMAP truly delivers on these promises in Notebook 10.